# Labeling the Dataset using the trained student

What we do:
1) Run the student model on the dataset of inputs
2) Analyse the student dataset and label it


Student: Qwen2.5-1.5B-Instruct

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [2]:
from core.types import *
from core.utils.huggingface_client import HuggingFaceClient
from core.utils.huggingface_inference_client import HuggingFaceInferenceClient
from core.utils.ollama_inference_client import OllamaInferenceClient
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.utils.doom_game_state import DoomGameState, MonsterType, WeaponName, AimedAtType
from sklearn.cluster import DBSCAN
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Iterable
from pathlib import Path
from ollama import ChatResponse
from openai.types.responses import Response as OpenAIResponse
from transformers import AutoTokenizer

import os
import json
import numpy as np
import pandas as pd

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
inputs = LLMCommandingInput.load_inputs(
    path=Path("data/inputs/inputs.json"),
    gstype=DoomGameState
)

inputs_lookup = {inp.id: inp for inp in inputs}

In [5]:
# For now, only extract inputs to label
selected_inputs = [
    inp
    for inp in inputs
    if inp.selected_for_labelling
]

print(f"Selected inputs: {len(selected_inputs)}/{len(inputs)}")

Selected inputs: 50/2872


In [13]:
student_client = OllamaInferenceClient[LLMCommandingInput, LLMCommandingOutput](
    model="qwen-commanding-q4",
    max_output_tokens=800,
    temperature=0.0,
)

🚀 Initialized OllamaInferenceClient for qwen-commanding-q4


In [14]:
# Prepare Prompt (same as training)
system_prompt = "You are a game command parser that converts natural language commands into DSL instructions."

In [15]:
def format_input(inp: LLMCommandingInput) -> str:
    game_state = inp.game_state.state.to_prompt_ready()
    command = inp.user_command.command.command
    return f"Game State:\n{game_state}\nCommand:\n{command}"


def parse_output(response: ChatResponse, input_id: str, latency: float) -> LLMCommandingOutput:
    return LLMCommandingOutput(
        input_id=input_id,
        actions=response.message.content,
        reason=None,
        latency=latency,
    )


def get_id(gse: LLMCommandingInput, idx: int) -> str:
    return gse.id

In [16]:
print(system_prompt)
print(format_input(inputs[3]))

You are a game command parser that converts natural language commands into DSL instructions.
Game State:
AIMED_AT:
  type: Wall
  distance: 330.86
  interactable: yes

MONSTERS (count=0):

INVENTORY:
  current_slot: 2
  weapons:
    - (1, Fist, 0)
    - (2, Pistol, 50)
Command:
Go press that switch ahead


In [17]:
outputs = student_client.process(
    dataset=selected_inputs,
    system_prompt=system_prompt,
    tools = [], # No tools at level 3
    format_input=format_input,
    parse_output=parse_output,
    get_id=get_id,
)

🔄 Processing 50 items sequentially


Processing items: 100%|██████████| 50/50 [00:10<00:00,  4.55it/s]


✅ Completed: 50/50 successful


In [18]:
# Clustering was already made earlier, so inputs are already partitioned.
# Now, considering this is just a test of the Teacher's quality (to save time):
# - Having retrieved the LLMCommandingOutputs, I can just prepare the csv
# - This time, every row in the csv should be set to be evaluated.
# For the full run: check if selected. (MAKE SURE TO CHANGE FILE NAME SO THAT I DO NOT HAVE TO REVALUATE IF THEY ALREADY CORRECT)

rows = []
for idx, output in enumerate(outputs):
    inp = inputs_lookup[output.input_id]

    row = LLMCommandingLabelledDataPoint(
        input_id=inp.id,
        game_state=inp.game_state.state.to_prompt_ready(),
        command=inp.user_command.command.command,
        command_intent=inp.user_command.command.intent,
        command_explicitness=inp.user_command.command.explicitness,
        command_atomicity=float(inp.user_command.command.atomicity),
        command_contextuality=float(inp.user_command.command.contextuality),
        game_actions=output.actions.__str__(),
        latency=output.latency,
        reason_if_failed=output.reason,
        cluster_id=inp.cluster_id,
        selected_for_labelling=inp.selected_for_labelling,
    )

    rows.append(asdict(row))

df = pd.DataFrame(rows)

In [19]:
output_path = Path("data/outputs/selected-data-training-qwen-1.5b-ollama-q4.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)